# Prep: clean + feature-select the datasets

Runs cleaning and feature selection for every dataset in `DATASETS` in one go, then **Save Version**. The reproduce/novel notebooks add this notebook's output as an input and skip cleaning.

Setup: turn Internet ON and attach the uploaded raw-data dataset (the folder with `nsl_kdd/`, `unsw_nb15/`, `cic-ids-2017/`, `cds-ids-2018/`). It is read in place; nothing is copied.

In [ ]:
DATASETS = ["nsl_kdd", "unsw_nb15", "unsw_nb15_nodedup", "cic_ids2017", "cse_cic_ids2018"]

In [ ]:
import glob, os, shutil, subprocess, sys

WORK = "/kaggle/working"
os.environ["RIS_DATA_PROCESSED"] = f"{WORK}/data/processed"
os.environ["RIS_RESULTS"] = f"{WORK}/results"

# Clone (or update) the repo; needs Internet turned on in notebook settings
REPO = f"{WORK}/RIS-M26"
if os.path.isdir(REPO):
    subprocess.run(["git", "-C", REPO, "pull"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/Sahoo-Achyutananda/RIS-M26.git", REPO], check=True)
SRC = f"{REPO}/src"

# The uploaded raw folder (containing nsl_kdd/, unsw_nb15/, cic-ids-2017/, cds-ids-2018/) is read
# in place from /kaggle/input: nothing is copied, so saved versions stay small.
RAW_MARKERS = {
    "nsl_kdd": "nsl_kdd/KDDTrain+*",
    "unsw_nb15": "unsw_nb15/UNSW_NB15_training-set.csv",
    "unsw_nb15_nodedup": "unsw_nb15/UNSW_NB15_training-set.csv",
    "cic_ids2017": "cic*2017/*WorkingHours*.csv",
    "cse_cic_ids2018": "c*2018/*-2018.csv",
}

def point_at_raw(datasets):
    roots, missing = set(), []
    for ds in datasets:
        hits = glob.glob(f"/kaggle/input/**/{RAW_MARKERS[ds]}", recursive=True)
        if hits:
            roots |= {os.path.dirname(os.path.dirname(h)) for h in hits}
        else:
            missing.append(ds)
    if missing or len(roots) != 1:
        for root, _, files in os.walk("/kaggle/input"):
            for fn in files:
                print("  ", os.path.join(root, fn))
        raise AssertionError(f"raw data not found for {missing}, or found in several places: {roots}")
    os.environ["RIS_DATA_RAW"] = roots.pop()
    print("raw data folder:", os.environ["RIS_DATA_RAW"])

point_at_raw(DATASETS)

In [ ]:
!pip install -q imbalanced-learn

In [ ]:
for ds in DATASETS:
    for script in ("preprocess.py", "feature_selection.py"):
        print("=== " + ds + ": " + script + " ===", flush=True)
        proc = subprocess.run([sys.executable, "-u", script, "--dataset", ds],
                              cwd=SRC, env=os.environ, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        print(proc.stdout[-2500:])
        assert proc.returncode == 0, f"{ds}: {script} failed"

In [ ]:
import pandas as pd
for ds in DATASETS:
    d = pd.read_csv(f"{WORK}/data/processed/{ds}/dataset_final.csv")
    print(f"{ds}: {d.shape}, attack share {d['label'].mean():.3f}")